# 🧾 Arabic Invoice Key-Info Extractor  *(Multi-Image + Evaluation)*
**Pipeline:** Detection Model → Arabic OCR → Fine-tuned LiLT → Structured JSON → **Evaluation Metrics**

---
### ⚙️ Before running:
1. **Enable GPU** → `Session options` → Accelerator → **GPU T4 x2**
2. **Enable Internet** → Session options → Internet → **On**
3. Configure paths in **Cell 3** (model checkpoint, invoice images, ground-truth)
---

## 1. Install Dependencies

In [ ]:
%%capture
!pip install easyocr --quiet
!pip install transformers==4.40.0 --quiet
!pip install sentencepiece --quiet
!pip install opencv-python-headless --quiet
!pip install jiwer --quiet          # fast CER / WER
print('✅ Dependencies installed')

## 2. Imports & GPU Check

In [ ]:
import json, os, zipfile, time
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict

import numpy as np
from PIL import Image
import torch
import easyocr
from transformers import AutoTokenizer, AutoModelForTokenClassification

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU found — check Accelerator setting in Session options')

## 3. Configuration

Upload your files via **➕ Add data** and set the paths below.

| Variable | Description |
|---|---|
| `MODEL_ZIP` | Zipped fine-tuned LiLT checkpoint |
| `INVOICE_IMAGES` | **List** of invoice image paths to process |
| `GROUND_TRUTH` | Dict mapping image filename → expected field values (for evaluation) |
| `DETECTION_URL/KEY` | YOLOv8 OBB detection API credentials |

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────

MODEL_ZIP = '/kaggle/input/your-dataset/lilt-finetuned.zip'

# List of invoice images to process (add as many as you like)
INVOICE_IMAGES: List[str] = [
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_01.jpg',
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_02.jpg',
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_03.jpg',
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_04.png',
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_05.png',
    '/kaggle/input/datasets/mzamziane/test-set-final/test_image_06.png',
]

# Ground truth for evaluation — set to {} to skip evaluation
# Keys are filename stems (no extension), values are dicts of expected fields.
# Example:
#   'invoice_001': {'invoice_number': 'INV-2024-001', 'total': '1500.00', ...}
GROUND_TRUTH: Dict[str, Dict[str, str]] = {
    'test_image_01': {
        'invoice_number': 'INV-0805-2024',
        'invoice_date':   '30-06-1980',
        'vendor_name':    'آل جعفر PLC',
        'recipient_name': 'مؤيد آل ربيع',
        'subtotal':       '3782.05',
        'tax':            '567.31',
        'total':          '4349.36'
    },
    'test_image_02': {
        'invoice_number': 'INV-4048-2024',
        'invoice_date':   '2013-01-01',
        'vendor_name':    'بقشان-آل جعفر',
        'recipient_name': 'الدكتورة ايمان آل عواض',
        'subtotal':       '5888.14',
        'tax':            '883.22',
        'total':          '6771.36'
    },
    'test_image_03': {
        'invoice_number': 'INV-2308-2024',
        'invoice_date':   '01-06-2015',
        'vendor_name':    'العليان-آل علي',
        'recipient_name': 'آل الشيخ-آل رفيع',
        'subtotal':       '3285.66',
        'tax':            '492.85',
        'total':          '3778.51'
    },
    'test_image_04': {
        'invoice_number': '36',
        'invoice_date':   '2022-12-14',
        'vendor_name':    'شركة ورود تولين التجارية',
        'recipient_name': 'ارجوان للزهور',
        'subtotal':       '191646.00',
        'tax':            '28746.90',
        'total':          '203292.90'
    },
    'test_image_05': {
        'invoice_number': '384',
        'invoice_date':   '2023/03/05',
        'vendor_name':    'شركة بوابة الصين المحدودة',
        'recipient_name': 'صباح بنت عبدالله احمد المغربي',
        'subtotal':       '6792.00',
        'tax':            '1018.80',
        'total':          '7810.80'
    },
    'test_image_06': {
        'invoice_number': '3439',
        'invoice_date':   '2023/02/27',
        'vendor_name':    'شركة بوابة الصين المحدودة',
        'recipient_name': 'صباح بنت عبدالله احمد المغربي',
        'subtotal':       '9186.52',
        'tax':            '1377.98',
        'total':          '10564.50'
    }
}

# YOLOv8 detection API
DETECTION_URL  = "https://predict-6a00dbf6ca8643714403-dproatj77a-no.a.run.app/predict"
DETECTION_KEY  = "ul_274eff7dd7124e218cc53f37913d17f55b829092"
DETECTION_ARGS = {"conf": 0.25, "iou": 0.7, "imgsz": 640}

MODEL_DIR = Path('/kaggle/input/models/mzamziane/lilt-fine-tuned-final/transformers/default/1/best-lilt-model')

if not MODEL_DIR.exists():
    print('Extracting model checkpoint...')
    with zipfile.ZipFile(MODEL_ZIP, 'r') as zf:
        zf.extractall('/kaggle/working/')
    extracted = list(Path('/kaggle/working/').glob('lilt*'))
    print(f'Extracted to: {extracted}')

print('✅ Config ready')
print(f'   Model dir   : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'   Images      : {len(INVOICE_IMAGES)} file(s)')
for p in INVOICE_IMAGES:
    print(f'     {Path(p).name}  exists={Path(p).exists()}')

## 4. Label Map

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(str(MODEL_DIR))
LABEL_MAP = {int(k): v for k, v in cfg.id2label.items()}

print(f'Found {len(LABEL_MAP)} labels:')
for idx, label in sorted(LABEL_MAP.items()):
    print(f'  {idx:3d}  {label}')

## 5. Pipeline Code

In [ ]:
import requests
import cv2

@dataclass
class DetectedRegion:
    """One bbox from the YOLOv8 OBB detection API."""
    detection_id: str
    x: int; y: int; width: int; height: int
    confidence: float

    @classmethod
    def from_prediction(cls, pred: dict, idx: int) -> 'DetectedRegion':
        box = pred['box']
        xs = [box['x1'], box['x2'], box['x3'], box['x4']]
        ys = [box['y1'], box['y2'], box['y3'], box['y4']]
        x0, y0 = int(min(xs)), int(min(ys))
        x1, y1 = int(max(xs)), int(max(ys))
        return cls(detection_id=f"det_{idx:04d}", x=x0, y=y0,
                   width=x1-x0, height=y1-y0, confidence=pred['confidence'])

    def as_xyxy(self): return self.x, self.y, self.x+self.width, self.y+self.height

    def normalize(self, img_w, img_h):
        return [max(0,min(1000,int(self.x/img_w*1000))),
                max(0,min(1000,int(self.y/img_h*1000))),
                max(0,min(1000,int((self.x+self.width)/img_w*1000))),
                max(0,min(1000,int((self.y+self.height)/img_h*1000)))]


@dataclass
class OcrToken:
    text: str; bbox_norm: list; bbox_pixel: list; confidence: float


def call_detection_api(image_path: str):
    print(f'  Calling detection API for {Path(image_path).name}...')
    with open(image_path, 'rb') as f:
        response = requests.post(
            DETECTION_URL,
            headers={'Authorization': f'Bearer {DETECTION_KEY}'},
            data=DETECTION_ARGS, files={'file': f})
    response.raise_for_status()
    raw = response.json()
    img_data    = raw['images'][0]
    h, w        = img_data['shape']
    img_meta    = {'width': w, 'height': h}
    predictions = img_data['results']
    regions = [DetectedRegion.from_prediction(p, i)
               for i, p in enumerate(predictions) if p.get('name') == 'text']
    regions.sort(key=lambda r: (r.y // 50, -r.x))
    print(f'  → {len(regions)} text regions')
    return regions, img_meta


def preprocess_for_ocr(image: Image.Image) -> np.ndarray:
    img = np.array(image.convert('L'))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)


def ocr_regions(image, regions, img_w, img_h, reader, min_conf=0.2):
    img_np      = preprocess_for_ocr(image)
    raw_results = reader.readtext(img_np, detail=1, paragraph=False)
    print(f'  easyocr raw detections: {len(raw_results)}')
    tokens = []
    for (pts, text, conf) in raw_results:
        text = text.strip()
        if not text or conf < min_conf:
            continue
        xs = [p[0] for p in pts]; ys = [p[1] for p in pts]
        ox0,oy0,ox1,oy1 = int(min(xs)),int(min(ys)),int(max(xs)),int(max(ys))
        ocx,ocy = (ox0+ox1)/2, (oy0+oy1)/2
        best_region, best_dist = None, float('inf')
        for region in regions:
            rx0,ry0,rx1,ry1 = region.as_xyxy()
            if ox0<rx1 and ox1>rx0 and oy0<ry1 and oy1>ry0:
                dist = ((ocx-(rx0+rx1)/2)**2+(ocy-(ry0+ry1)/2)**2)**0.5
                if dist < best_dist:
                    best_dist, best_region = dist, region
        if best_region:
            norm_bbox  = best_region.normalize(img_w, img_h)
            pixel_bbox = list(best_region.as_xyxy())
        else:
            norm_bbox  = [max(0,min(1000,int(ox0/img_w*1000))),
                          max(0,min(1000,int(oy0/img_h*1000))),
                          max(0,min(1000,int(ox1/img_w*1000))),
                          max(0,min(1000,int(oy1/img_h*1000)))]
            pixel_bbox = [ox0,oy0,ox1,oy1]
        tokens.append(OcrToken(text=text, bbox_norm=norm_bbox,
                               bbox_pixel=pixel_bbox, confidence=conf))
    tokens.sort(key=lambda t: (t.bbox_pixel[1]//50, -t.bbox_pixel[0]))
    return tokens


def run_lilt(tokens, model_path, label_map, device):
    tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
    model     = AutoModelForTokenClassification.from_pretrained(model_path).to(device)
    model.eval()
    words = [t.text for t in tokens]; boxes = [t.bbox_norm for t in tokens]
    encoding = tokenizer(words, is_split_into_words=True, return_tensors='pt',
                         truncation=True, max_length=512, padding='max_length')
    word_ids = encoding.word_ids(batch_index=0)
    aligned_boxes = [boxes[wid] if wid is not None else [0,0,0,0] for wid in word_ids]
    bbox_tensor   = torch.tensor([aligned_boxes], dtype=torch.long).to(device)
    with torch.no_grad():
        outputs = model(input_ids=encoding['input_ids'].to(device),
                        attention_mask=encoding['attention_mask'].to(device),
                        bbox=bbox_tensor)
    preds = outputs.logits.argmax(-1)[0].tolist()
    seen, results = set(), []
    for idx, wid in enumerate(word_ids):
        if wid is None or wid in seen: continue
        seen.add(wid)
        t = tokens[wid]
        results.append({'text': t.text, 'label': label_map.get(preds[idx],'O'),
                        'bbox_pixel': t.bbox_pixel, 'confidence': t.confidence})
    return results


def build_invoice_dict(tagged):
    entities, cur_base, cur_parts = {}, None, []
    def flush():
        if cur_base and cur_parts:
            entities.setdefault(cur_base, []).append(' '.join(cur_parts))
    for tok in tagged:
        lbl, txt = tok['label'], tok['text']
        if lbl == 'O':
            flush(); cur_base, cur_parts = None, []
        elif lbl.startswith('B-'):
            flush(); cur_base, cur_parts = lbl[2:], [txt]
        elif lbl.startswith('I-'):
            base = lbl[2:]
            if base == cur_base: cur_parts.append(txt)
            else: flush(); cur_base, cur_parts = base, [txt]
    flush()
    return {k:(v[0] if len(v)==1 else v) for k,v in entities.items()}


print('✅ Pipeline functions defined')

## 6. Run Pipeline — Multiple Images

In [ ]:
print('Initialising easyocr (Arabic + English)...')
reader = easyocr.Reader(['ar', 'en'], gpu=(DEVICE == 'cuda'))

all_results   = []   # list of dicts: {filename, invoice, tagged_tokens, latency_s}
latency_times = []

for img_path in INVOICE_IMAGES:
    img_name = Path(img_path).name
    print(f'\n──────────────────────────────────────────')
    print(f'  Processing: {img_name}')
    print( '──────────────────────────────────────────')

    try:
        t_start = time.perf_counter()

        # Step 1 — Load + Detect
        image        = Image.open(img_path).convert('RGB')
        IMG_W, IMG_H = image.size
        regions, _   = call_detection_api(img_path)

        # Step 2 — OCR
        tokens = ocr_regions(image, regions, IMG_W, IMG_H, reader, min_conf=0.3)
        print(f'  Tokens extracted: {len(tokens)}')

        # Step 3 — LiLT NER
        tagged_tokens = run_lilt(tokens, str(MODEL_DIR), LABEL_MAP, DEVICE)

        # Step 4 — Build structured output
        invoice = build_invoice_dict(tagged_tokens)

        t_end    = time.perf_counter()
        latency  = t_end - t_start
        latency_times.append(latency)

        all_results.append({
            'filename'     : img_name,
            'invoice'      : invoice,
            'tagged_tokens': tagged_tokens,
            'latency_s'    : round(latency, 3),
        })

        print(f'\n  Extracted fields:')
        for k, v in invoice.items():
            print(f'    {k:<30} {v}')
        print(f'  ⏱  Latency: {latency:.2f}s')

    except Exception as e:
        print(f'  ❌ Error processing {img_name}: {e}')

# Save batch output
out_path = '/kaggle/working/batch_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump([{k:v for k,v in r.items() if k!='tagged_tokens'}
               for r in all_results], f, ensure_ascii=False, indent=2)

avg_lat = sum(latency_times)/len(latency_times) if latency_times else 0
print(f'\n✅ Processed {len(all_results)}/{len(INVOICE_IMAGES)} invoices')
print(f'   Avg latency : {avg_lat:.2f}s/image')
print(f'   Results saved → {out_path}')

## 7. Evaluation Metrics

Computes the metrics described in the thesis:
- **CER / WER** — character- and word-level edit-distance rates
- **Precision, Recall, F1** — token-classification quality per label
- **Exact Match Accuracy** — percentage of fields extracted perfectly
- **Normalized Field Accuracy** — Levenshtein-based soft score per field
- **IoU** — bounding-box localisation quality (requires ground-truth boxes)
- **mAP** — mean Average Precision over detected field classes
- **Latency** — per-image wall-clock time summary

> **To run this cell** you must populate `GROUND_TRUTH` in Cell 3 with expected field values.
> IoU / mAP also require `GROUND_TRUTH_BOXES` (see the cell below).

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
#  EVALUATION METRICS
# ════════════════════════════════════════════════════════════════════════════════
import re
import unicodedata
from jiwer import cer, wer
from collections import defaultdict

# ── helpers ────────────────────────────────────────────────────────────────────

def levenshtein(s1: str, s2: str) -> int:
    """Standard dynamic-programming Levenshtein distance."""
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], prev if s1[i-1]==s2[j-1] else 1+min(prev, dp[j], dp[j-1])
    return dp[n]


def normalize_field(s: str) -> str:
    """Unicode-normalize, strip whitespace, lower-case for comparison."""
    s = unicodedata.normalize('NFC', str(s))
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s


# ────────────────────────────────────────────────────────────────────────────────
# 1. CER / WER  (OCR transcript quality)
# ────────────────────────────────────────────────────────────────────────────────
# Requires OCR ground-truth text per image.  Populate the dict below if available.
# Format:  'invoice_001' → 'full ground-truth OCR text of that invoice'
OCR_GROUND_TRUTH: Dict[str, str] = {
    # 'invoice_001': 'شركة الأمل  رقم الفاتورة INV-2024-001 ...',
}

print('=' * 60)
print('  1. CER / WER  (OCR quality)')
print('=' * 60)

cer_scores, wer_scores = [], []

for result in all_results:
    stem = Path(result['filename']).stem
    if stem not in OCR_GROUND_TRUTH:
        continue
    # Build the predicted transcript from tagged tokens
    predicted_text = ' '.join(
        tok['text'] for tok in result['tagged_tokens']
    )
    reference_text = OCR_GROUND_TRUTH[stem]
    c = cer(reference_text, predicted_text)
    w = wer(reference_text, predicted_text)
    cer_scores.append(c); wer_scores.append(w)
    print(f'  {result["filename"]:30s}  CER={c:.4f}  WER={w:.4f}')

if cer_scores:
    print(f'  {"──────────────────────────────────────────":40s}')
    print(f'  {"Mean":30s}  CER={sum(cer_scores)/len(cer_scores):.4f}  '
          f'WER={sum(wer_scores)/len(wer_scores):.4f}')
else:
    print('  (No OCR ground-truth provided — skipped)')


# ────────────────────────────────────────────────────────────────────────────────
# 2. Precision, Recall, F1  (NER token classification)
# ────────────────────────────────────────────────────────────────────────────────
# Requires token-level ground-truth labels. Populate per image.
# Format: 'invoice_001' → list of (text, true_label) tuples
NER_GROUND_TRUTH: Dict[str, List] = {
    # 'invoice_001': [('شركة', 'B-vendor_name'), ('الأمل', 'I-vendor_name'), ...],
}

print()
print('=' * 60)
print('  2. Precision / Recall / F1  (NER token classification)')
print('=' * 60)

tp_per_label: Dict[str, int] = defaultdict(int)
fp_per_label: Dict[str, int] = defaultdict(int)
fn_per_label: Dict[str, int] = defaultdict(int)

for result in all_results:
    stem = Path(result['filename']).stem
    if stem not in NER_GROUND_TRUTH:
        continue
    gt_labels  = [lbl for _, lbl in NER_GROUND_TRUTH[stem]]
    pred_labels = [tok['label'] for tok in result['tagged_tokens']]
    for p, g in zip(pred_labels, gt_labels):
        if p == g and p != 'O':   tp_per_label[g] += 1
        elif p != g:
            if p != 'O': fp_per_label[p] += 1
            if g != 'O': fn_per_label[g] += 1

all_labels = sorted(set(list(tp_per_label)+list(fp_per_label)+list(fn_per_label)))

if all_labels:
    print(f'  {"Label":<35} {"Prec":>6} {"Rec":>6} {"F1":>6}')
    print(f'  {"─"*60}')
    macro_f1 = []
    for lbl in all_labels:
        tp = tp_per_label[lbl]; fp = fp_per_label[lbl]; fn = fn_per_label[lbl]
        prec = tp/(tp+fp) if (tp+fp) else 0
        rec  = tp/(tp+fn) if (tp+fn) else 0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0
        macro_f1.append(f1)
        print(f'  {lbl:<35} {prec:>6.3f} {rec:>6.3f} {f1:>6.3f}')
    print(f'  {"─"*60}')
    print(f'  {"Macro avg":<35} {"":>6} {"":>6} {sum(macro_f1)/len(macro_f1):>6.3f}')
else:
    print('  (No NER ground-truth provided — skipped)')


# ────────────────────────────────────────────────────────────────────────────────
# 3. Exact Match Accuracy + Normalized Field Accuracy  (field extraction)
# ────────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  3. Field Extraction — Exact Match & Normalized Accuracy')
print('=' * 60)

if not GROUND_TRUTH:
    print('  (No GROUND_TRUTH provided — skipped)')
else:
    field_exact:      Dict[str, List[int]]   = defaultdict(list)
    field_norm_acc:   Dict[str, List[float]] = defaultdict(list)
    total_fields = 0; exact_total = 0

    for result in all_results:
        stem = Path(result['filename']).stem
        if stem not in GROUND_TRUTH:
            continue
        gt    = GROUND_TRUTH[stem]
        pred  = result['invoice']
        for field, gt_val in gt.items():
            gt_n   = normalize_field(gt_val)
            pred_n = normalize_field(pred.get(field, ''))
            exact  = int(gt_n == pred_n)
            dist   = levenshtein(gt_n, pred_n)
            norm   = max(0.0, 1 - dist / max(len(gt_n), 1))
            field_exact[field].append(exact)
            field_norm_acc[field].append(norm)
            total_fields += 1; exact_total += exact

    all_fields = sorted(field_exact.keys())
    print(f'  {"Field":<30} {"Exact":>7} {"NormAcc":>9}')
    print(f'  {"─"*50}')
    for fld in all_fields:
        e = sum(field_exact[fld])/len(field_exact[fld])
        n = sum(field_norm_acc[fld])/len(field_norm_acc[fld])
        print(f'  {fld:<30} {e:>7.1%} {n:>9.4f}')

    if total_fields:
        print(f'  {"─"*50}')
        print(f'  {"Micro avg (all fields)":<30} '
              f'{exact_total/total_fields:>7.1%} '
              f'{sum(sum(v) for v in field_norm_acc.values())/total_fields:>9.4f}')


# ────────────────────────────────────────────────────────────────────────────────
# 4. IoU  (bounding-box localisation)
# ────────────────────────────────────────────────────────────────────────────────
# Format: 'invoice_001' → list of {'label': ..., 'bbox': [x0,y0,x1,y1]}
GROUND_TRUTH_BOXES: Dict[str, List[Dict]] = {
    # 'invoice_001': [
    #     {'label': 'invoice_number', 'bbox': [100, 50, 400, 80]},
    #     {'label': 'total',          'bbox': [300,500, 500,530]},
    # ],
}

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0

print()
print('=' * 60)
print('  4. IoU  (bounding-box localisation quality)')
print('=' * 60)

iou_per_label: Dict[str, List[float]] = defaultdict(list)

for result in all_results:
    stem = Path(result['filename']).stem
    if stem not in GROUND_TRUTH_BOXES: continue
    gt_boxes = GROUND_TRUTH_BOXES[stem]
    for gt in gt_boxes:
        base_lbl = gt['label']
        # Find best-matching predicted token for this label
        pred_boxes = [tok['bbox_pixel'] for tok in result['tagged_tokens']
                      if tok['label'].replace('B-','').replace('I-','') == base_lbl]
        if not pred_boxes: continue
        best_iou = max(compute_iou(gt['bbox'], pb) for pb in pred_boxes)
        iou_per_label[base_lbl].append(best_iou)

if iou_per_label:
    IOU_THRESH = 0.5
    print(f'  IoU threshold = {IOU_THRESH}')
    print(f'  {"Label":<30} {"Mean IoU":>10} {"Recall@0.5":>12}')
    print(f'  {"─"*55}')
    for lbl in sorted(iou_per_label):
        ious     = iou_per_label[lbl]
        mean_iou = sum(ious)/len(ious)
        recall   = sum(1 for v in ious if v >= IOU_THRESH) / len(ious)
        print(f'  {lbl:<30} {mean_iou:>10.4f} {recall:>12.1%}')
    all_ious = [v for vals in iou_per_label.values() for v in vals]
    print(f'  {"─"*55}')
    print(f'  {"Overall mean IoU":<30} {sum(all_ious)/len(all_ious):>10.4f}')
else:
    print('  (No GROUND_TRUTH_BOXES provided — skipped)')


# ────────────────────────────────────────────────────────────────────────────────
# 5. mAP  (mean Average Precision over field classes)
# ────────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  5. mAP  (mean Average Precision)')
print('=' * 60)

# For mAP we need confidence scores per prediction matched against GT boxes.
# We use each tagged token's OCR confidence as the detector confidence.

def compute_ap(recalls, precisions):
    """All-point interpolation AP."""
    r = [0.0] + list(recalls) + [1.0]
    p = [1.0] + list(precisions) + [0.0]
    # Make precision monotonically decreasing
    for i in range(len(p)-2, -1, -1):
        p[i] = max(p[i], p[i+1])
    ap = sum((r[i+1]-r[i]) * p[i+1] for i in range(len(r)-1))
    return ap

if GROUND_TRUTH_BOXES:
    IOU_MAP_THRESH = 0.5
    class_preds: Dict[str, List] = defaultdict(list)   # label → [(conf, tp)]
    class_gt_count: Dict[str, int] = defaultdict(int)

    for result in all_results:
        stem = Path(result['filename']).stem
        if stem not in GROUND_TRUTH_BOXES: continue
        gt_boxes   = GROUND_TRUTH_BOXES[stem]
        matched_gt = set()
        # Sort predictions by confidence descending
        preds_sorted = sorted(result['tagged_tokens'],
                              key=lambda t: t['confidence'], reverse=True)
        for tok in preds_sorted:
            base = tok['label'].replace('B-','').replace('I-','')
            if base == 'O': continue
            pb   = tok['bbox_pixel']
            conf = tok['confidence']
            best_iou, best_idx = 0.0, -1
            for gi, gt in enumerate(gt_boxes):
                if gt['label'] != base or gi in matched_gt: continue
                iou = compute_iou(pb, gt['bbox'])
                if iou > best_iou: best_iou, best_idx = iou, gi
            tp = int(best_iou >= IOU_MAP_THRESH and best_idx >= 0)
            if tp: matched_gt.add(best_idx)
            class_preds[base].append((conf, tp))
        for gt in gt_boxes:
            class_gt_count[gt['label']] += 1

    ap_scores = {}
    print(f'  IoU threshold = {IOU_MAP_THRESH}')
    print(f'  {"Class":<30} {"AP":>8}  (GT count)')
    print(f'  {"─"*50}')
    for cls in sorted(class_preds):
        preds_sorted = sorted(class_preds[cls], reverse=True)
        tp_cumsum  = np.cumsum([tp for _, tp in preds_sorted])
        fp_cumsum  = np.cumsum([1-tp for _, tp in preds_sorted])
        n_gt       = class_gt_count.get(cls, 1)
        recalls    = tp_cumsum / n_gt
        precisions = tp_cumsum / (tp_cumsum + fp_cumsum)
        ap         = compute_ap(recalls.tolist(), precisions.tolist())
        ap_scores[cls] = ap
        print(f'  {cls:<30} {ap:>8.4f}  (n={n_gt})')
    if ap_scores:
        mAP = sum(ap_scores.values()) / len(ap_scores)
        print(f'  {"─"*50}')
        print(f'  {"mAP":<30} {mAP:>8.4f}')
else:
    print('  (No GROUND_TRUTH_BOXES provided — skipped)')


# ────────────────────────────────────────────────────────────────────────────────
# 6. Latency summary
# ────────────────────────────────────────────────────────────────────────────────
print()
print('=' * 60)
print('  6. Processing Time / Latency')
print('=' * 60)

if latency_times:
    lats = latency_times
    print(f'  {"Metric":<25} {"Value":>10}')
    print(f'  {"─"*38}')
    print(f'  {"N images":<25} {len(lats):>10}')
    print(f'  {"Mean latency (s)":<25} {sum(lats)/len(lats):>10.3f}')
    print(f'  {"Min  latency (s)":<25} {min(lats):>10.3f}')
    print(f'  {"Max  latency (s)":<25} {max(lats):>10.3f}')
    std = (sum((x - sum(lats)/len(lats))**2 for x in lats)/len(lats))**0.5
    print(f'  {"Std  latency (s)":<25} {std:>10.3f}')
    print(f'  {"Throughput (img/s)":<25} {len(lats)/sum(lats):>10.3f}')
else:
    print('  (No latency data available)')

print()
print('✅ Evaluation complete.')

## 8. Visualise Detected Regions (per image)

In [ ]:
from PIL import ImageDraw
import matplotlib.pyplot as plt

LABEL_COLORS = {
    'invoice_number' : '#e74c3c', 'invoice_date'   : '#e67e22',
    'vendor_name'    : '#2ecc71', 'recipient_name' : '#3498db',
    'total'          : '#9b59b6', 'tax'            : '#1abc9c',
    'subtotal'       : '#f39c12', 'due_date'       : '#e91e63',
}
DEFAULT_COLOR = '#95a5a6'

for result in all_results:
    img_path = next(p for p in INVOICE_IMAGES
                    if Path(p).name == result['filename'])
    image    = Image.open(img_path).convert('RGB')
    vis      = image.copy()
    draw     = ImageDraw.Draw(vis)

    for tok in result['tagged_tokens']:
        if tok['label'] == 'O': continue
        x0, y0, x1, y1 = tok['bbox_pixel']
        base  = tok['label'].replace('B-','').replace('I-','')
        color = LABEL_COLORS.get(base, DEFAULT_COLOR)
        draw.rectangle([x0,y0,x1,y1], outline=color, width=3)
        draw.text((x0, max(0,y0-18)), tok['label'], fill=color)

    scale = 800 / max(vis.size)
    vis_s = vis.resize((int(vis.width*scale), int(vis.height*scale)))

    plt.figure(figsize=(12, 16))
    plt.imshow(vis_s); plt.axis('off')
    plt.title(result['filename'], fontsize=13)
    plt.tight_layout()

    out_vis = f'/kaggle/working/annotated_{result["filename"]}'
    plt.savefig(out_vis, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out_vis}')